# Spark con Python — Versión sencilla
### Notas de estudiantes

**Antes de empezar:**

Ejecuta las celdas **en orden, de arriba hacia abajo** (Shift + Enter en cada una).
La primera se demora ~2 minutos porque descarga el motor de Spark. Las demás son rápidas.

**Los datos:** 5.180 registros de notas de estudiantes en 5 cursos.
**La pregunta que vamos a responder:** ¿cuál es el promedio de notas por área y por curso, y cuántos estudiantes aprueban?

El dataset viene sucio a propósito: tiene **filas duplicadas** y **notas faltantes**.


---
## Paso 0 · Instalar PySpark

| | Qué es |
|---|---|
| **Apache Spark** | El motor. Escrito en Scala, corre sobre Java. |
| **PySpark** | La librería de Python que le da órdenes al motor. |

`pip install pyspark` **descarga los dos**: el paquete trae el motor empaquetado adentro
(~300 MB, por eso se demora). Lo único aparte es **Java**, y Colab ya lo trae.


In [ ]:
!pip install -q pyspark
!java -version


---
## Paso 1 · Cargar los archivos

Sube **`notas.csv`** y **`cursos.csv`** con el panel de la izquierda (icono de carpeta 📁 → subir),
o ejecuta la celda de abajo para escogerlos desde tu computador.


In [ ]:
from google.colab import files
subidos = files.upload()   # escoge notas.csv y cursos.csv
print('\nCargados:', list(subidos.keys()))


In [ ]:
!ls -lh *.csv
!head -4 notas.csv
!cat cursos.csv


---
## Paso 2 · Crear la SparkSession

La `SparkSession` es la **puerta de entrada**: el objeto que representa tu conexión con el motor.
Sin ella no se puede hacer nada.

### La única línea que cambia entre practicar y producción

```python
.master("local[*]")                       # aquí mismo, con todos los núcleos
.master("spark://servidor-empresa:7077")  # en un clúster real
```

Todo el resto del código es **idéntico**. Aprendes con 5.000 filas en tu portátil
y el mismo código corre con 500 GB en la nube.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("NotasEstudiantes")
         .master("local[*]")        # 'local' = mi máquina | '[*]' = todos los núcleos
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")   # silencia los mensajes de advertencia

print('Spark version:', spark.version)


---
## Paso 3 · Leer los datos

Aquí uso `inferSchema=True`, que le dice a Spark *"adivina tú los tipos de cada columna"*.
Es lo más cómodo para explorar.

> **Ojo para producción:** `inferSchema` obliga a Spark a leer el archivo **dos veces**
> (una para adivinar y otra para cargar). Con 5.000 filas ni se nota; con 500 GB estarías
> pagando el doble de lectura. Allá se declara el esquema a mano.


In [ ]:
notas  = spark.read.csv("notas.csv",  header=True, inferSchema=True)
cursos = spark.read.csv("cursos.csv", header=True, inferSchema=True)

print('Listo. Pero ojo: todavía NO se ha leído ni un byte del archivo.')


### El momento clave: la evaluación perezosa

La celda de arriba terminó **instantáneamente**. ¿Por qué?

Porque `spark.read.csv(...)` es una **transformación**: Spark solo anotó *"cuando me toque,
hay que leer este archivo"*. No leyó nada.

La celda de abajo tiene `.show()` y `.count()`, que son **acciones**. Ahí sí arranca de verdad
y por eso tarda un poco más. **Compara los tiempos de las dos celdas.**


In [ ]:
notas.show(5)                                # ACCIÓN → ejecuta
notas.printSchema()                          # los tipos de cada columna
print('Filas leídas:', notas.count())        # ACCIÓN → ejecuta

cursos.show()


> **`printSchema()`** es el equivalente de `df.dtypes` en pandas, y siempre debe ser
> lo primero que revisas después de leer un archivo.


---
## Paso 4 · Diagnosticar: ¿qué tan sucios están los datos?

Antes de limpiar hay que medir el daño. Dos preguntas: **¿cuántos nulos?** y **¿cuántos duplicados?**


In [ ]:
# ¿Cuántos nulos hay en cada columna?
# Se lee de adentro hacia afuera: por cada columna c, cuenta las filas donde el valor es nulo.
notas.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
              for c in notas.columns]).show()

# ¿Cuántos duplicados?
print('Filas totales :', notas.count())
print('IDs distintos :', notas.select('id_registro').distinct().count())
print('Duplicados    :', notas.count() - notas.select('id_registro').distinct().count())


---
## Paso 5 · Limpiar

### ⚠️ Aquí está la decisión importante de todo el ejercicio

Hay que rellenar las notas faltantes. La tentación es usar el promedio general de todos los cursos.
**Sería un error.**

Mira los números de la celda de abajo: Cálculo tiene un promedio de ~2.9 e Inglés de ~4.3.
Si relleno una nota faltante de Cálculo con el promedio general (~3.7), le estoy **regalando
casi un punto** a ese estudiante y contaminando el resultado del curso.

> **Regla:** cuando los grupos tienen comportamientos distintos, se imputa **por grupo**, no en global.


In [ ]:
# Primero comprobamos que los cursos SÍ se comportan distinto
notas.groupBy("id_curso").agg(
    F.round(F.avg("nota"), 2).alias("promedio"),
    F.count("nota").alias("con_nota")
).orderBy("id_curso").show()

print('Promedio GENERAL:', round(notas.agg(F.avg('nota')).first()[0], 2))


Ahora sí, la limpieza. Aparecen dos herramientas nuevas:

**`Window.partitionBy("id_curso")`** — es como un `groupBy` que **no colapsa las filas**.
Un `groupBy` convierte 5.000 filas en 5. Una ventana deja las 5.000 filas intactas, pero cada una
puede consultar un cálculo hecho sobre su propio grupo. Es exactamente lo que necesitamos:
conservar cada registro, pero rellenar con el promedio de *su* curso.

**`F.coalesce(a, b)`** — viene de SQL y significa *"devuélveme el primer valor que no sea nulo"*.
Si la nota existe la deja; si es nula, mete el promedio del curso.

> **Orden importante:** el `dropDuplicates` va **antes** de calcular el promedio. Si lo dejaras
> después, las 180 filas duplicadas pesarían doble y sesgarían el cálculo.


In [ ]:
from pyspark.sql import Window

w = Window.partitionBy("id_curso")      # 'agrupa por curso, pero sin colapsar las filas'

notas_limpias = (notas
    .dropDuplicates(["id_registro"])                    # 1. quita duplicados
    .withColumn("nota",
        F.coalesce(F.col("nota"),                        # 2. si la nota existe, la deja...
                   F.round(F.avg("nota").over(w), 2)))   #    si no, el promedio de SU curso
    .filter(F.col("horas_estudio") >= 0))               # 3. descarta datos imposibles

print('ANTES  :', notas.count(),         'filas')
print('DESPUÉS:', notas_limpias.count(), 'filas')

# Comprobamos que ya no quedan nulos
print('Nulos restantes:', notas_limpias.filter(F.col('nota').isNull()).count())


---
## Paso 6 · Transformar: crear columnas nuevas

`withColumn("nombre", expresión)` crea una columna. Si el nombre ya existe, la reemplaza.

**Los DataFrames son inmutables**: esto no modifica `notas_limpias`, devuelve uno nuevo.
Por eso lo guardo en otra variable. No existe `inplace=True`.

`F.when(...).otherwise(...)` es el **`CASE WHEN` de SQL**.


In [ ]:
notas_tr = (notas_limpias
    .withColumn("estado",
        F.when(F.col("nota") >= 3.0, "Aprobó").otherwise("Reprobó"))
    .withColumn("dedicacion",
        F.when(F.col("horas_estudio") >= 12, "Alta")
         .when(F.col("horas_estudio") >= 6,  "Media")
         .otherwise("Baja")))

notas_tr.select("estudiante", "id_curso", "nota", "estado",
                "horas_estudio", "dedicacion").show(8)


> ⚠️ **Nunca escribas funciones de Python para aplicar fila por fila (UDFs).**
> El motor está en Java: una UDF obliga a que cada fila cruce de Java a Python y vuelva,
> y el rendimiento se cae al piso. Si existe una función `F.` que hace lo que necesitas, úsala.


---
## Paso 7 · Unir tablas (join)

`notas` tiene el `id_curso` pero no sabe cómo se llama el curso ni a qué área pertenece.
Esa información está en `cursos`. Unirlas es lo que en bases de datos sería un `JOIN`.

⚠️ Un `join` provoca un **shuffle**: Spark mueve datos entre trabajadores para juntar las filas
que coinciden. Es la operación más costosa de Spark y la causa nº 1 de que un proceso se demore.


In [ ]:
completo = notas_tr.join(cursos, on="id_curso", how="left")
completo.select("estudiante", "curso", "area", "nota", "estado").show(5)


---
## Paso 8 · Agregar: responder la pregunta

**La pregunta:** ¿cuál es el promedio por curso y qué porcentaje aprueba?

`groupBy(...).agg(...)` es el patrón central de toda la analítica en Spark.
`.alias()` le pone nombre a la columna resultante (si no, queda como `avg(nota)`).

El truco del porcentaje: `F.when(condición, 1).otherwise(0)` convierte la condición en unos y ceros,
y el **promedio de unos y ceros es la proporción**. Multiplicado por 100 da el porcentaje.


In [ ]:
resumen = (completo
    .groupBy("area", "curso")
    .agg(F.round(F.avg("nota"), 2).alias("promedio"),
         F.count("*").alias("estudiantes"),
         F.round(F.avg(F.when(F.col("estado") == "Aprobó", 1).otherwise(0)) * 100, 1)
          .alias("pct_aprobacion"))
    .orderBy(F.desc("promedio")))

resumen.show(truncate=False)


### Otra pregunta: ¿estudiar más sirve?

Agrupamos por el nivel de dedicación y vemos el promedio de cada uno.


In [ ]:
(completo.groupBy("dedicacion")
 .agg(F.round(F.avg("nota"), 2).alias("promedio"),
      F.count("*").alias("estudiantes"))
 .orderBy(F.desc("promedio"))
 .show())


---
## Paso 9 · Lo mismo, pero en SQL

Si registras el DataFrame como una vista, puedes consultarlo con SQL puro.
**Pasa por el mismo optimizador y rinde exactamente igual**: es cuestión de gusto.


In [ ]:
completo.createOrReplaceTempView("notas")

spark.sql("""
    SELECT area,
           ROUND(AVG(nota), 2) AS promedio,
           COUNT(*)            AS estudiantes
    FROM notas
    GROUP BY area
    ORDER BY promedio DESC
""").show()


---
## Paso 10 · El puente hacia pandas

Spark **no grafica**. Para visualizar hay que traer el resultado a pandas con `.toPandas()`.

⚠️ `.toPandas()` trae **todo** a la memoria de una sola máquina. Sobre 500 GB revienta el proceso.
Solo se usa sobre resultados **ya agregados y pequeños** — aquí el resumen son 5 filas.

Este es el flujo real de la industria:

```
Spark procesa 500 GB  →  devuelve un resumen de 2 MB  →  pandas lo grafica
```


In [ ]:
import matplotlib.pyplot as plt

por_curso = (completo.groupBy("curso")
             .agg(F.round(F.avg("nota"), 2).alias("promedio"))
             .orderBy(F.desc("promedio"))
             .toPandas())        # <-- aquí cruzamos de Spark a pandas

print(type(por_curso))           # ahora sí es un DataFrame de pandas
display(por_curso)

por_curso.plot(kind="barh", x="curso", y="promedio",
               legend=False, color="#E8622C", figsize=(8, 4))
plt.title("Promedio de notas por curso")
plt.xlabel("Nota promedio")
plt.tight_layout()
plt.show()


---
## Paso 11 · Guardar el resultado

### ¿Por qué Parquet y no CSV?

| | CSV | Parquet |
|---|---|---|
| Tipos de dato | Se pierden (todo es texto) | Se conservan |
| Tamaño | Grande | 5–10× más pequeño |
| Lectura selectiva | Lee el archivo entero | Lee solo las columnas que pides |

Parquet es **columnar**: guarda por columna en vez de por fila. Es el formato estándar del data lake.

> Verás que `salida/resumen` no es un archivo sino una **carpeta** con pedazos adentro
> (`part-00000...`). Es normal: cada partición se escribe por separado y Spark las junta al leer.


In [ ]:
resumen.write.mode("overwrite").parquet("salida/resumen")
!ls -R salida/

# Y así se lee de vuelta
spark.read.parquet("salida/resumen").show()


---
## Paso 12 · Experimento: comprueba tú misma la pereza

Encadenamos varias transformaciones **sin ninguna acción** y medimos el tiempo.
Después ejecutamos la acción y medimos otra vez.


In [ ]:
import time

t0 = time.time()
plan = (notas
        .dropDuplicates(["id_registro"])
        .filter(F.col("horas_estudio") > 5)
        .groupBy("id_curso")
        .agg(F.avg("nota").alias("promedio"))
        .orderBy(F.desc("promedio")))
print(f'Solo TRANSFORMACIONES: {time.time() - t0:.4f} s   (no se procesó nada)')

t0 = time.time()
plan.show()                     # <-- la ACCIÓN
print(f'Con la ACCIÓN        : {time.time() - t0:.4f} s   (aquí sí trabajó)')


### La prueba de que el optimizador funciona

`.explain()` imprime el plan que armó Catalyst. Busca la línea que empieza con **`FileScan csv`**
y mira **qué columnas aparecen entre corchetes**.

El archivo tiene 5 columnas, pero el plan solo va a leer las que la consulta necesita.
Nadie se lo pidió: el optimizador lo dedujo solo porque vio la receta completa antes de cocinar.
**Eso es para lo que sirve la evaluación perezosa.**


In [ ]:
plan.explain()


---
## Paso 13 · Cerrar la sesión


In [ ]:
spark.stop()


---
## Chuleta: pandas → Spark

| Quiero... | pandas | Spark |
|---|---|---|
| Leer un CSV | `pd.read_csv(f)` | `spark.read.csv(f, header=True)` |
| Primeras filas | `df.head()` | `df.show(5)` |
| Ver los tipos | `df.dtypes` | `df.printSchema()` |
| Contar filas | `len(df)` | `df.count()` |
| Elegir columnas | `df[["a","b"]]` | `df.select("a","b")` |
| Filtrar | `df[df.a > 10]` | `df.filter(F.col("a") > 10)` |
| Columna nueva | `df["c"] = df.a * 2` | `df.withColumn("c", F.col("a") * 2)` |
| Condicional | `np.where(...)` | `F.when(...).otherwise(...)` |
| Agrupar | `df.groupby("a")["b"].mean()` | `df.groupBy("a").agg(F.avg("b"))` |
| Promedio por grupo sin colapsar | `df.groupby("a").transform("mean")` | `F.avg("b").over(Window.partitionBy("a"))` |
| Unir | `df.merge(d2, on="k")` | `df.join(d2, on="k")` |
| Ordenar | `df.sort_values("a")` | `df.orderBy("a")` |
| Quitar duplicados | `df.drop_duplicates()` | `df.dropDuplicates()` |
| Guardar | `df.to_csv(f)` | `df.write.parquet(f)` |

**No existe en Spark:** índice, `.loc`, `.iloc`, `inplace=True`, `iterrows()`.
